# Niveaux fins de la taxonomie d'injection — evaluation GROUP-aware

Deux questions, sur le 3e jeu augmente (`prompt_injection_aegis.csv`, 7927 instances) :

1. **delta (target_delta)** : peut-on predire la couche de defense ciblee, en evaluation honnete (split groupe par template) ?
2. **technique fine** : des features plus riches (hashing char n-grams, embeddings de phrases) debloquent-elles la prediction de famille / technique ?

**Methodo cle** : tout est evalue en `GroupKFold(groups=template_group)` — aucun template des deux cotes du split. Un split aleatoire donnerait des scores >0.95 illusoires (memorisation de template).

**Securite (content filter)** : ce notebook ne charge que des **matrices numeriques** pre-calculees par `data/generators/build_aegis_text_features.py` (features de surface, hashing char n-grams stateless, embeddings MiniLM frozen) + les labels. Le texte des attaques n'y transite jamais. Les features hashing/embeddings sont **independantes par echantillon** (pas d'ajustement sur le corpus), donc une matrice unique est sans fuite sous GroupKFold.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from scipy import sparse
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

D = Path('..') / 'data' / 'aegis_text_features'
lab = pd.read_csv(D / 'labels.csv')
Xs = np.load(D / 'surface.npy')                       # 15 surface features
Xh = sparse.load_npz(D / 'hash.npz')                  # char n-grams (3-5), 2**18 dims
Xe = np.load(D / 'emb.npy') if (D / 'emb.npy').exists() else None  # MiniLM 384d
g = lab['template_group'].values
print('samples', len(lab), '| surface', Xs.shape, '| hashing', Xh.shape,
      '| emb', None if Xe is None else Xe.shape)
print('templates (groupes):', lab['template_group'].nunique())

In [ ]:
def gkf_f1(X, y, clf, mask=None, n=5):
    """F1 macro en GroupKFold sur template_group (split honnete, anti-fuite)."""
    yv = y.astype(str).values
    gg = g
    if mask is not None:
        X = X[mask]; yv = yv[mask]; gg = g[mask]
    splits = GroupKFold(n).split(X, yv, gg)
    return cross_val_score(clf, X, yv, cv=splits, scoring='f1_macro', n_jobs=-1).mean()

RF = lambda: RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)
LR = lambda: LogisticRegression(max_iter=300)
SV = lambda: LinearSVC()  # lineaire pour le hashing haute dimension

## Partie 1 — delta (couche de defense ciblee)

4 classes (delta0..delta3), tres desequilibrees (delta1/delta2 dominent). Hasard ~0.25.

In [ ]:
m = lab['target_delta'].notna().values
y = lab['target_delta']
rows = [['surface (15)', gkf_f1(Xs, y, RF(), m)],
        ['hashing char n-grams', gkf_f1(Xh, y, SV(), m)]]
if Xe is not None:
    rows.append(['embeddings MiniLM', gkf_f1(Xe, y, LR(), m)])
pd.DataFrame(rows, columns=['features', 'f1_macro (GroupKFold)']).round(3)

**Resultat de reference (mesure)** : surface 0.534 | hashing 0.567 | embeddings 0.563.

delta **generalise** (~2x le hasard) et les features riches n'apportent qu'un gain marginal (~+0.03). C'est la cible fine la plus exploitable telle quelle.

## Partie 2 — les features riches debloquent-elles famille / technique ?

In [ ]:
out = []
for tgt in ['family_l2', 'technique_l3']:
    y = lab[tgt]; m = y.notna().values
    row = {'cible': tgt, 'classes': int(y[m].nunique()),
           'surface': gkf_f1(Xs, y, RF(), m),
           'hashing': gkf_f1(Xh, y, SV(), m)}
    if Xe is not None:
        row['embeddings'] = gkf_f1(Xe, y, LR(), m)
    out.append(row)
pd.DataFrame(out).round(3)

**Resultat de reference (mesure)** :

| cible | classes | surface | hashing | embeddings |
|---|---|---|---|---|
| family_l2 | 17 | 0.177 | **0.283** | 0.263 |
| technique_l3 | 81 | 0.037 | 0.034 | 0.025 |

- **famille** : les features riches aident nettement (+60% relatif). Les familles ont plusieurs templates -> la generalisation inter-template est possible.
- **technique fine** : aucune feature n'aide. Pourquoi ? -> partie 3.

## Partie 3 — pourquoi la technique fine ne se debloque pas : le cold-start

Sous GroupKFold, une technique qui n'a **qu'un seul template** est toujours absente de l'entrainement quand son template est en test -> impredictible, **quelles que soient les features**. C'est une limite de **structure de donnees**, pas de richesse de features.

In [ ]:
tpt = lab.groupby('technique_l3')['template_group'].nunique()
print('techniques avec 1 seul template :', int((tpt == 1).sum()), '/', tpt.size)
print('techniques avec >=2 templates    :', int((tpt >= 2).sum()))

multi = set(tpt[tpt >= 2].index)
m = lab['technique_l3'].isin(multi).values
y = lab['technique_l3']
ref = {'sous-ensemble apprenable (>=2 templates)': int(m.sum()),
       'classes': len(multi),
       'surface': round(gkf_f1(Xs, y, RF(), m), 3)}
if Xe is not None:
    ref['embeddings'] = round(gkf_f1(Xe, y, LR(), m), 3)
ref

**Resultat de reference (mesure)** : meme restreint aux 17 techniques ayant >=2 templates, surface 0.191 / embeddings 0.119 — toujours bas. Le verrou est le **nombre de templates distincts par technique**, pas les features.

## Conclusions

1. **delta generalise** (~0.53-0.57) — cible fine exploitable ; features riches marginales.
2. **famille** : les features riches (hashing/embeddings) **debloquent** un gain reel (0.18 -> 0.28).
3. **technique fine** : ni hashing ni embeddings ne debloquent — c'est un **cold-start** (64/81 techniques = 1 template). Le levier n'est pas les features mais **plus de templates distincts par technique** (authoring AEGIS) ou un regroupement en familles.
4. **Toujours** evaluer en `GroupKFold(template_group)` : le split aleatoire surestime massivement (memorisation de template).

**Prochaines etapes** : (a) viser >=3 templates distincts par technique prioritaire pour rendre la technique apprenable ; (b) exploiter delta + famille comme cibles fines fiables ; (c) tester des embeddings specialises domaine medical.